# Phase 2a: segment long clips with forced alignment

Cuts every train and dev clip longer than 30 s into pieces of at most 30 s, each with its own exact text (`cyasr/segment.py`). The test split is never touched.

Run on a Colab T4 (Runtime > Change runtime type > T4 GPU). Before starting, run `bash scripts/make_colab_bundle.sh` locally and upload the new `cyasr_bundle.zip` to Google Drive (`MyDrive/cyasr/`), replacing the old one.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BUNDLE = '/content/drive/MyDrive/cyasr/cyasr_bundle.zip'
DRIVE_SEGMENTS = '/content/drive/MyDrive/cyasr/segments'

!rm -rf /content/cyasr && mkdir -p /content/cyasr
!unzip -q {BUNDLE} -d /content/cyasr
%cd /content/cyasr
!ls cyasr/segment.py  # fails if the bundle is the old one
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# torch is already installed on Colab; this adds the aligner and the other requirements
!pip install -q -r requirements.txt

In [ ]:
# Smoke test on 3 long clips, written to a separate folder so it never mixes with the real output.
# First run downloads the MMS aligner model (~1.2 GB).
!python -m cyasr.segment --split train --max-clips 3 --out-dir /content/segtest

In [ ]:
# Full run. Each split is copied to Drive as soon as it finishes, so a disconnect doesn't lose it.
for split in ['train', 'dev']:
    !python -m cyasr.segment --split {split}
    !mkdir -p {DRIVE_SEGMENTS} && cp -r data/segments/* {DRIVE_SEGMENTS}/

In [ ]:
# Quality summary per split
import pandas as pd

seg = pd.read_csv('data/segments/segments.tsv', sep='\t')
for split, df in seg.groupby('split'):
    aligned = df[df['align_score'].notna()]
    odd_rate = df[(df['chars_per_sec'] < 3) | (df['chars_per_sec'] > 30)]
    print(f"{split}: {df['audio_id'].nunique()} clips -> {len(df)} pieces, "
          f"{df['duration_s'].sum() / 3600:.2f} h, longest piece {df['duration_s'].max():.1f} s")
    print(f"  aligned pieces: {len(aligned)}, odd speaking rate: {len(odd_rate)}")
    print(aligned.nsmallest(5, 'align_score')[['id', 'duration_s', 'align_score', 'text']].to_string(index=False))
    print()